In [1]:
import cryo
import polars as pl
import binascii
import web3
import json
from eth_abi import decode
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import pandas as pd

In [2]:
# The multical cantract address, but we also need ABI
MULTICALL3_ADDRESS = '0xcA11bde05977b3631167028862bE2a173976CA11'
MULTICALL3_ABI=json.loads('[{"inputs":[{"internalType":"bool","name":"requireSuccess","type":"bool"},{"components":[{"internalType":"address","name":"target","type":"address"},{"internalType":"bytes","name":"callData","type":"bytes"}],"internalType":"struct Multicall3.Call[]","name":"calls","type":"tuple[]"}],"name":"tryAggregate","outputs":[{"components":[{"internalType":"bool","name":"success","type":"bool"},{"internalType":"bytes","name":"returnData","type":"bytes"}],"internalType":"struct Multicall3.Result[]","name":"returnData","type":"tuple[]"}],"stateMutability":"payable","type":"function"}]')

In [3]:
# Contract Addresses
UNIV3_USDC_ETH='0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640'
UNIV3_WBTC_ETH = '0xCBCdF9626bC03E24f779434178A73a0B4bad62eD'
UNIV2_ETH_USDC= '0xB4e16d0168e52d35CaCD2c6185b44281Ec28C9Dc'
SUSHIV2_USDC_ETH='0x397FF1542f962076d0BFE58eA045FfA2d347ACa0'

In [4]:
# Function Signatures 4 bytes
getBlocknumber_4b = '42cbb15c'
getBloclTimestamp_4b= '0f28c97d'
getReserves_4b = '0902f1ac'
slot0_4b = '3850c7bd'

In [5]:
# Functions
def bytes_to_hexstr(b: any) -> str:
    if isinstance(b,list):
        return [bytes_to_hexstr(a) for a in b]
    return '0x' + b.hex()

def decode_outputdata_uniV2V3_price(b: bytes) -> list[float]:
    aggregated_data_uniV2V3 = decode(['(bool,bytes)[]'], b)[0]

    # UNI-V2
    ethusdc_reserves_raw = aggregated_data_uniV2V3[0]
    [eth_bal, usdc_bal, time ] = decode(['uint112','uint112','uint32'], ethusdc_reserves_raw[1])
    eth_usdc_price_v2 = (eth_bal / usdc_bal)*1e12

    # SUSHI-V2
    ethusdc_reserves_raw = aggregated_data_uniV2V3[0]
    [eth_bal, usdc_bal, time ] = decode(['uint112','uint112','uint32'], ethusdc_reserves_raw[1])
    eth_usdc_price_sushi = (eth_bal / usdc_bal)*1e12

    # UNI-V3
    # slot0():
    # sqrtPriceX96 uint160, tick int24, observationIndex uint16, observationCardinality uint16, observationCardinalityNext uint16, feeProtocol uint8, unlocked bool
    # 'uint160', 'int24', 'uint16', 'uint16', 'uint16', 'uint8', 'bool'
    usdc_eth_slot0_raw = aggregated_data_uniV2V3[1]
    usdc_eth_slot0_sqrt_ratioX96 = decode(['uint160', 'int24', 'uint16', 'uint16', 'uint16', 'uint8', 'bool'], usdc_eth_slot0_raw[1])[0]
    usdc_eth_price = usdc_eth_slot0_sqrt_ratioX96**2 / 2**192 /1e12
    eth_usdc_price_v3 = 1/usdc_eth_price
    # print(f"WBTC/ETH: { wbtc_eth_price}")
    # print(f"ETH/WBTC: { eth_wbtc_price}")

    # Timestamp
    timestamp_raw = aggregated_data_uniV2V3[-1]
    timestamp = int(timestamp_raw[1].hex(),16)
    
    return [eth_usdc_price_v2, eth_usdc_price_v3, eth_usdc_price_sushi, timestamp]

def decode_outputdata_uniV3_price(b: bytes) -> list[float]:
    aggregated_data_uniV3 = decode(['(bool,bytes)[]'], b)[0]

    # UNI-V3
    # slot0():
    # sqrtPriceX96 uint160, tick int24, observationIndex uint16, observationCardinality uint16, observationCardinalityNext uint16, feeProtocol uint8, unlocked bool
    # 'uint160', 'int24', 'uint16', 'uint16', 'uint16', 'uint8', 'bool'
    usdc_eth_slot0_raw = aggregated_data_uniV3[1]
    usdc_eth_slot0_sqrt_ratioX96 = decode(['uint160', 'int24', 'uint16', 'uint16', 'uint16', 'uint8', 'bool'], usdc_eth_slot0_raw[1])[0]
    usdc_eth_price = usdc_eth_slot0_sqrt_ratioX96**2 / 2**192 /1e12
    eth_usdc_price_v3 = 1/usdc_eth_price
    # print(f"WBTC/ETH: { wbtc_eth_price}")
    # print(f"ETH/WBTC: { eth_wbtc_price}")

    # Timestamp
    timestamp_raw = aggregated_data_uniV3[-1]
    timestamp = int(timestamp_raw[1].hex(),16)
    
    return [eth_usdc_price_v3, timestamp]

In [6]:
# web3 instance, function from web3py
w3 = web3.Web3()
m3 = w3.eth.contract(address = MULTICALL3_ADDRESS, abi=MULTICALL3_ABI)

In [7]:
# Arguments fro the tryAggregate Fuunction
aggregate_calldata = [
    [
        UNIV2_ETH_USDC,
        f'0x{getReserves_4b}',
    ],
    [
        UNIV3_USDC_ETH,
        f'0x{slot0_4b}',
    ],
    [
        SUSHIV2_USDC_ETH,
        f'0x{getReserves_4b}',
    ],
    [
        MULTICALL3_ADDRESS,
        f'0x{getBloclTimestamp_4b}'
    ],
]

In [8]:
aggregate_calldata

[['0xB4e16d0168e52d35CaCD2c6185b44281Ec28C9Dc', '0x0902f1ac'],
 ['0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640', '0x3850c7bd'],
 ['0x397FF1542f962076d0BFE58eA045FfA2d347ACa0', '0x0902f1ac'],
 ['0xcA11bde05977b3631167028862bE2a173976CA11', '0x0f28c97d']]

In [11]:
# Generate calldate (the input) via m3 Multicall3 encode ABIfor cryo -Hex format
calldata = m3.encode_abi("tryAggregate", args=[False, aggregate_calldata])

In [12]:
#  Call data passed to cryo collect 
cryo_kwargs = {
    'rpc': 'https://eth.merkle.io',
    'blocks': ['-10:latest'], 
}
            
eth_call_uni_df = cryo.collect(
    'eth_calls',
    to_address = [MULTICALL3_ADDRESS],
    call_data=[calldata],
     output_format="polars",
    **cryo_kwargs,
)

In [13]:
# Binary Format
output_data=eth_call_uni_df['output_data'][0]

In [14]:
# It returns binary output: here we separate in 2 parts
aggregated_data_uniV2V3 = decode(['(bool,bytes)[]'], output_data)[0]

In [15]:
# Function decode_outputdata_uniV2V3_price uses eth_abi.decode to get decimals values from binary format
prices = [decode_outputdata_uniV2V3_price(x) for x in eth_call_uni_df['output_data'].to_list()]

In [16]:
prices

[[2570.166369321173, 2575.8819306706664, 2570.166369321173, 1747386239],
 [2570.166369321173, 2575.8819306706664, 2570.166369321173, 1747386251],
 [2570.1636387389126, 2575.8819306706664, 2570.1636387389126, 1747386263],
 [2570.1636387389126, 2575.8816953472788, 2570.1636387389126, 1747386275],
 [2570.1636387389126, 2575.8486208384224, 2570.1636387389126, 1747386287],
 [2572.2729032965667, 2578.7348727139183, 2572.2729032965667, 1747386299],
 [2572.2729032965667, 2578.49918642487, 2572.2729032965667, 1747386311],
 [2573.13724977815, 2579.579899882553, 2573.13724977815, 1747386323],
 [2573.13724977815, 2579.6120267954307, 2573.13724977815, 1747386335],
 [2577.235939625078, 2583.6885313717135, 2577.235939625078, 1747386347]]